# 2. Consumer & Classifier

## Purpose

This notebook is the **core analytical stage** of the pipeline. It:

1. Consumes HTTP request events from the Kafka topic `http.requests.raw`
2. Runs the **trained Random Forest classifier** (from CSIC 2010 data) to detect attacks
3. Applies **rule-based pattern detection** to identify the specific attack type
4. Maps findings to the **MITRE ATT&CK framework**
5. Emits a **Jaeger span** for each stage (consume → classify → mitre → write)
6. Saves all results to `pipeline_data/classified_results.csv`

**Run this notebook in parallel with notebook 1.**

---

| Stage | Jaeger Span |
|---|---|
| Kafka consume | `consume_http_request` |
| ML classification | `ml.classify` |
| Attack type detection | `rule.detect_attack_type` |
| MITRE lookup | `mitre.lookup` |
| CSV write | `storage.write_csv` |

In [1]:
import json
import csv
import os
import sys
import re
import random
from datetime import datetime, timezone

from kafka import KafkaConsumer

from opentelemetry import trace
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor
from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
from opentelemetry.trace import SpanKind, TraceFlags, SpanContext, NonRecordingSpan, set_span_in_context
import uuid

# ── Add project src/ to path ───────────────────────────────────────────────
sys.path.insert(0, '/home/jovyan/work')
from src.classifier import AttackClassifier
from src.features import extract_features
from src.mitre_mapper import detect_attack_type, get_mitre_info

def _normalize_url(url: str) -> str:
    """Strip scheme+host so the URL matches the training format (path only).
    Training data went through preprocessor._extract_path() which strips
    'http://host' — we must do the same here to avoid feature mismatch."""
    url = re.sub(r'\s+HTTP/\S+$', '', url.strip())   # remove trailing HTTP/1.1
    url = re.sub(r'^https?://[^/]+', '', url)          # remove http://localhost:8080
    return url or '/'

# ── Configuration ──────────────────────────────────────────────────────────
KAFKA_BOOTSTRAP = os.environ.get('KAFKA_BOOTSTRAP', 'kafka:9092')
JAEGER_ENDPOINT = os.environ.get('JAEGER_ENDPOINT', 'jaeger:4317')
TOPIC           = 'http.requests.raw'
GROUP_ID        = 'http-classifier'
OUTPUT_CSV      = '/home/jovyan/work/pipeline_data/classified_results.csv'
MODEL_PATH      = '/home/jovyan/work/models/classifier.joblib'

print(f'Kafka : {KAFKA_BOOTSTRAP}')
print(f'Jaeger: {JAEGER_ENDPOINT}')
print(f'Output: {OUTPUT_CSV}')


Kafka : kafka:9092
Jaeger: jaeger:4317
Output: /home/jovyan/work/pipeline_data/classified_results.csv


In [2]:
# ── Load trained classifier ────────────────────────────────────────────────
clf = AttackClassifier.load(MODEL_PATH)
status = 'trained model' if clf.is_trained else 'rule-based fallback (no model found)'
print(f'Classifier loaded: {status}')

# ── OpenTelemetry / Jaeger setup ───────────────────────────────────────────
resource = Resource.create({'service.name': 'http-request-classifier'})
trace.set_tracer_provider(TracerProvider(resource=resource))
tracer = trace.get_tracer(__name__)
otlp_exporter = OTLPSpanExporter(endpoint=JAEGER_ENDPOINT, insecure=True)
trace.get_tracer_provider().add_span_processor(BatchSpanProcessor(otlp_exporter))

# ── Kafka consumer ─────────────────────────────────────────────────────────
consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    group_id=GROUP_ID,
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    value_deserializer=lambda v: json.loads(v.decode('utf-8')),
)

# ── CSV output ─────────────────────────────────────────────────────────────
os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
file_exists = os.path.exists(OUTPUT_CSV)
csv_file = open(OUTPUT_CSV, 'a', newline='')
FIELDNAMES = ['event_id', 'timestamp', 'method', 'url', 'label',
              'ml_label', 'confidence', 'attack_type',
              'mitre_id', 'mitre_name', 'mitre_tactic', 'mitre_url',
              'chain_detected']
writer = csv.DictWriter(csv_file, fieldnames=FIELDNAMES)
if not file_exists:
    writer.writeheader()

print('Consumer ready. Waiting for messages...')
print('Check Jaeger → http://localhost:16686  (service: http-request-classifier)')

Classifier loaded: trained model
Consumer ready. Waiting for messages...
Check Jaeger → http://localhost:16686  (service: http-request-classifier)


In [3]:
# -- Sanity check: verify classifier before consuming ──────────────────────
import importlib, src.features as feat_mod
importlib.reload(feat_mod)

test_cases = [
    {'method':'GET',  'url':'/tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=150&cantidad=2',
     'body':'', 'headers':{}, '_expected':'normal'},
    {'method':'POST', 'url':'/tienda1/publico/autenticar.jsp',
     'body':"correo=' OR '1'='1'--&password=x", 'headers':{}, '_expected':'anomalous'},
    {'method':'GET',  'url':'/tienda1/publico/buscar.jsp?id=5&nombre=Aceite+Oliva&precio=200&cantidad=1',
     'body':'', 'headers':{}, '_expected':'normal'},
]

print('Classifier sanity check:')
all_ok = True
for t in test_cases:
    req = {k: v for k, v in t.items() if not k.startswith('_')}
    result = clf.predict(req)
    ok = result['label'] == t['_expected']
    all_ok = all_ok and ok
    icon = '\u2705' if ok else '\u274c'
    print(f'  {icon} expected={t["_expected"]:<10} got={result["label"]:<10} conf={result["confidence"]:.0%}  {t["url"][:60]}')

if not all_ok:
    raise RuntimeError('Classifier sanity check failed — re-run notebook 0 first.')
print('Sanity check passed. Starting consumer...')


Classifier sanity check:
  ✅ expected=normal     got=normal     conf=97%  /tienda1/publico/anadir.jsp?id=3&nombre=Vino+Rioja&precio=15
  ✅ expected=anomalous  got=anomalous  conf=100%  /tienda1/publico/autenticar.jsp
  ✅ expected=normal     got=normal     conf=100%  /tienda1/publico/buscar.jsp?id=5&nombre=Aceite+Oliva&precio=
Sanity check passed. Starting consumer...


In [ ]:
# ── Main consume loop ──────────────────────────────────────────────────────
from collections import defaultdict
import time as _time

# Sliding-window attack chain detector
# Tracks attack types per session cookie within CHAIN_WINDOW seconds.
# Flags a session when it accumulates 2+ distinct attack types.
CHAIN_WINDOW = 120   # seconds
_session_log = defaultdict(list)   # cookie -> [(epoch_float, attack_type)]

def _check_chain(cookie, attack_type, ts_epoch):
    cutoff = ts_epoch - CHAIN_WINDOW
    _session_log[cookie] = [(t, a) for t, a in _session_log[cookie] if t >= cutoff]
    _session_log[cookie].append((ts_epoch, attack_type))
    types_seen = {a for _, a in _session_log[cookie]}
    return sorted(types_seen) if len(types_seen) >= 2 else None

processed = 0
attacks   = 0
chains    = 0

try:
    for msg in consumer:
        event    = msg.value
        event_id = event.get('event_id', str(uuid.uuid4()))

        trace_id = uuid.UUID(event_id).int
        parent_ctx = set_span_in_context(
            NonRecordingSpan(SpanContext(
                trace_id=trace_id,
                span_id=random.getrandbits(64),
                is_remote=True,
                trace_flags=TraceFlags(TraceFlags.SAMPLED),
                trace_state={},
            ))
        )

        with tracer.start_as_current_span('consume_http_request', context=parent_ctx, kind=SpanKind.CONSUMER) as span:
            span.set_attribute('event.id',    event_id)
            span.set_attribute('http.method', event.get('method', ''))
            span.set_attribute('http.url',    event.get('url', '')[:200])

            normalized_url = _normalize_url(event.get('url', ''))
            request = {
                'method':  event.get('method', ''),
                'url':     normalized_url,
                'body':    event.get('body', ''),
                'headers': {'user-agent': event.get('user_agent', '')},
            }

            with tracer.start_as_current_span('ml.classify') as ml_span:
                ml_result  = clf.predict(request)
                ml_label   = ml_result['label']
                confidence = round(ml_result['confidence'], 4)
                ml_span.set_attribute('ml.label',      ml_label)
                ml_span.set_attribute('ml.confidence', confidence)

            with tracer.start_as_current_span('rule.detect_attack_type') as rule_span:
                attack_type = detect_attack_type(request) if ml_label == 'anomalous' else None
                rule_span.set_attribute('attack.type', str(attack_type))

            with tracer.start_as_current_span('mitre.lookup') as mitre_span:
                mitre_info   = get_mitre_info(attack_type) if attack_type else {}
                mitre_id     = mitre_info.get('id', 'N/A')
                mitre_name   = mitre_info.get('name', 'N/A')
                mitre_tactic = mitre_info.get('tactic', 'N/A')
                mitre_url    = mitre_info.get('url', '')
                mitre_span.set_attribute('mitre.id',     mitre_id)
                mitre_span.set_attribute('mitre.tactic', mitre_tactic)
                mitre_span.set_attribute('mitre.url',    mitre_url)

            # ── Chain detection ────────────────────────────────────────────
            cookie = event.get('cookie', '')
            chain_detected = None
            with tracer.start_as_current_span('chain.detect') as chain_span:
                if ml_label == 'anomalous' and attack_type and cookie:
                    try:
                        ts_epoch = datetime.fromisoformat(
                            event.get('timestamp', datetime.now(timezone.utc).isoformat())
                        ).timestamp()
                    except Exception:
                        ts_epoch = _time.time()
                    chain_detected = _check_chain(cookie, attack_type, ts_epoch)
                chain_str = ','.join(chain_detected) if chain_detected else ''
                chain_span.set_attribute('chain.detected', chain_str)
                span.set_attribute('chain.detected', chain_str)

            span.set_attribute('ml.label',    ml_label)
            span.set_attribute('mitre.id',    mitre_id)
            span.set_attribute('attack.type', str(attack_type))

            with tracer.start_as_current_span('storage.write_csv'):
                writer.writerow({
                    'event_id':       event_id,
                    'timestamp':      event.get('timestamp', datetime.now(timezone.utc).isoformat()),
                    'method':         event.get('method', ''),
                    'url':            normalized_url[:200],
                    'label':          event.get('label', ''),
                    'ml_label':       ml_label,
                    'confidence':     confidence,
                    'attack_type':    attack_type or '',
                    'mitre_id':       mitre_id,
                    'mitre_name':     mitre_name,
                    'mitre_tactic':   mitre_tactic,
                    'mitre_url':      mitre_url,
                    'chain_detected': chain_str,
                })
                csv_file.flush()

        processed += 1
        if ml_label == 'anomalous':
            attacks += 1
        if chain_detected:
            chains += 1

        icon      = '\U0001f6a8' if ml_label == 'anomalous' else '\u2705'
        link      = f'  -> {mitre_url}' if mitre_url and ml_label == 'anomalous' else ''
        chain_tag = f'  [CHAIN: {" -> ".join(chain_detected)}]' if chain_detected else ''
        print(f'[{processed:>4}] {icon} {ml_label:<10} {confidence:.0%} conf | {attack_type or "-":<22} {mitre_id}{link}{chain_tag}')

except KeyboardInterrupt:
    print(f'\nConsumer stopped. Processed: {processed}, Attacks: {attacks}, Chains: {chains}')
finally:
    csv_file.close()
    consumer.close()


[   1] 🚨 anomalous  100% conf | parameter_tampering    T1565.001  -> https://attack.mitre.org/techniques/T1565/001/
[   2] ✅ normal     100% conf | -                      N/A
[   3] ✅ normal     100% conf | -                      N/A
[   4] 🚨 anomalous  100% conf | sql_injection          T1190  -> https://attack.mitre.org/techniques/T1190/
[   5] 🚨 anomalous  99% conf | parameter_tampering    T1565.001  -> https://attack.mitre.org/techniques/T1565/001/  [CHAIN: parameter_tampering -> sql_injection]
[   6] 🚨 anomalous  99% conf | parameter_tampering    T1565.001  -> https://attack.mitre.org/techniques/T1565/001/
[   7] ✅ normal     100% conf | -                      N/A
[   8] ✅ normal     100% conf | -                      N/A
[   9] ✅ normal     100% conf | -                      N/A
[  10] ✅ normal     100% conf | -                      N/A
[  11] ✅ normal     100% conf | -                      N/A
[  12] ✅ normal     100% conf | -                      N/A
[  13] ✅ normal     100% co